# Scenario 9: Auto-Tune & Playground — Iterative Prompt and Metric Improvement

## What You'll Learn

In this notebook you will:

1. **Set up a project** and create an evaluation dataset
2. **Use the Playground** (via SDK-equivalent experiments) to interactively test prompts
3. **Run Auto-Tune** to automatically optimize a prompt template
4. **Compare results** between your original and optimized prompts
5. **Submit scorer feedback** to improve custom metrics over time
6. **Re-run experiments** with the optimized prompt for final validation

---

## What is Auto-Tune?

**Auto-Tune** is Galileo's prompt optimization feature. Instead of manually tweaking prompts, Auto-Tune uses AI to analyze your current prompt's performance and suggest improvements — better instructions, clearer constraints, and more effective few-shot examples.

```
Your Prompt  →  Auto-Tune  →  Optimized Prompt
    │                              │
    ▼                              ▼
 Run Experiment              Run Experiment
    │                              │
    ▼                              ▼
 Baseline Scores            Improved Scores
    │                              │
    └──────── Compare ─────────────┘
```

## What is the Playground?

The **Playground** is Galileo's interactive prompt testing environment. In the Console UI, it lets you:
- Type a prompt and see the model's response immediately
- Adjust model settings (temperature, max_tokens) in real time
- View metric scores alongside responses

In this notebook, we approximate the Playground workflow using `run_experiment()` — the SDK equivalent of testing prompts interactively.

---

## Key Concepts

| Concept | What It Means |
|---------|---------------|
| **Auto-Tune** | AI-powered prompt optimization that suggests improvements to your prompt template |
| **Playground** | Interactive prompt testing environment in the Galileo Console |
| **Scorer Feedback** | A mechanism to provide feedback on metric scores (original vs. annotated) to improve custom scorers |
| **Prompt Optimization** | The process of iterating on prompts to improve quality, using data rather than intuition |

---

## Prerequisites

- A `.env` file with `GALILEO_API_KEY` and `OPENAI_API_KEY`
- Dependencies installed via `uv sync`
- An OpenAI integration configured in your Galileo console

### Step 0: Load Environment Variables

Same pattern as previous notebooks — load API keys from `.env`.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

for root in [Path.cwd(), Path.cwd().parent]:
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))

env_candidates = [Path.cwd() / '.env', Path.cwd().parent / '.env']
for candidate in env_candidates:
    if candidate.exists():
        load_dotenv(candidate)
        ENV_FILE = candidate
        break
else:
    raise FileNotFoundError('Could not find .env in the repo root or current directory')

print(f'Loaded credentials from {ENV_FILE}')

### Step 0b: Imports and Configuration

Key imports for this notebook:

- **`run_experiment`** — Runs a prompt template over a dataset and scores every row
- **`PromptRunSettings`** — Model configuration (model_alias, temperature, max_tokens)
- **`PromptTemplate`** / **`GlobalPromptTemplates`** — For creating reusable prompt templates
- **`requests`** — For Auto-Tune and scorer feedback REST API calls

In [ ]:
import os

import requests

from galileo import GalileoMetrics, Message, MessageRole, galileo_context
from galileo.config import GalileoPythonConfig
from galileo.datasets import create_dataset, delete_dataset, get_dataset
from galileo.experiments import PromptRunSettings, run_experiment
from galileo.log_streams import create_log_stream, get_log_stream
from galileo.projects import delete_project, get_project
from galileo.prompts import GlobalPromptTemplates

PROJECT_NAME = 'GalileoEval_S9_AutoTune'
LOG_STREAM_NAME = 'auto-tune-lab'
DATASET_NAME = 'auto-tune-eval-dataset'
dataset_id = None

## 1. Bootstrap Galileo and Create a Dataset

First we initialize a Galileo project and create an evaluation dataset. This dataset contains questions that we'll use to compare our original prompt against the Auto-Tuned version.

The dataset includes a mix of factual questions with reference answers — the `output` column serves as ground truth for metrics like `ground_truth_adherence`.

In [ ]:
galileo_context.init(project=PROJECT_NAME, log_stream=LOG_STREAM_NAME)

log_stream = get_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)
if not log_stream:
    log_stream = create_log_stream(name=LOG_STREAM_NAME, project_name=PROJECT_NAME)

config = GalileoPythonConfig.get()
logger = galileo_context.get_logger_instance()
project_id = getattr(logger, 'project_id', None)
log_stream_id = getattr(logger, 'log_stream_id', None)

if project_id and log_stream_id:
    project_url = f'{config.console_url}project/{project_id}'
    log_stream_url = f'{project_url}/log-streams/{log_stream_id}'
    print(project_url)
    print(log_stream_url)
else:
    print('Galileo context initialized')

# Create evaluation dataset
ds = get_dataset(name=DATASET_NAME)
if not ds:
    ds = create_dataset(
        name=DATASET_NAME,
        content=[
            {'input': 'What is photosynthesis?', 'output': 'Photosynthesis is the process by which plants convert sunlight, water, and CO2 into glucose and oxygen.'},
            {'input': 'Explain how a neural network learns.', 'output': 'A neural network learns by adjusting weights through backpropagation, minimizing the loss function over training data.'},
            {'input': 'What causes tides?', 'output': 'Tides are caused by the gravitational pull of the Moon and Sun on Earth\'s oceans.'},
            {'input': 'How does encryption work?', 'output': 'Encryption transforms plaintext into ciphertext using an algorithm and key, making data unreadable without the decryption key.'},
            {'input': 'What is the greenhouse effect?', 'output': 'The greenhouse effect is the trapping of heat by atmospheric gases like CO2, warming Earth\'s surface.'},
        ],
        project_name=PROJECT_NAME,
    )
dataset_id = ds.id
f'Dataset ready: {DATASET_NAME} ({dataset_id})'

## 2. Playground — Run a Baseline Experiment

The **Playground** in the Galileo Console lets you interactively test prompts and see results immediately. Here we approximate that workflow using `run_experiment()` — the SDK equivalent.

### What the Playground Offers (Console UI)

1. **Interactive prompt editor** — Type your system prompt and user message, see the response instantly
2. **Side-by-side comparison** — Test two prompts at once and compare outputs
3. **Real-time metrics** — See correctness, adherence, and other scores as you iterate
4. **Model switching** — Try the same prompt with different models (GPT-4, Claude, etc.)

### SDK Equivalent

We create a prompt template and run it over our dataset. This gives us a **baseline** to compare against after Auto-Tune optimization.

In [ ]:
BASELINE_TEMPLATE_NAME = 'auto-tune-baseline-template'

templates = GlobalPromptTemplates()
baseline_template = templates.get(name=BASELINE_TEMPLATE_NAME)
if baseline_template is None:
    baseline_template = templates.create(
        name=BASELINE_TEMPLATE_NAME,
        template=[
            Message(role=MessageRole.system, content='Answer the question accurately in 1-2 sentences.'),
            Message(role=MessageRole.user, content='{{input}}'),
        ],
        project_name=PROJECT_NAME,
    )

baseline_run = run_experiment(
    experiment_name='auto-tune-baseline',
    dataset=DATASET_NAME,
    prompt_template=baseline_template,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.ground_truth_adherence,
    ],
    prompt_settings=PromptRunSettings(model_alias='GPT-4o mini', temperature=0.3, max_tokens=150),
    project=PROJECT_NAME,
)
baseline_run['message']

## 3. Auto-Tune — Optimize the Prompt

**Auto-Tune** analyzes your prompt's performance on the dataset and suggests an optimized version. It considers:

- Which test cases scored lowest (where the prompt struggles)
- Common patterns in low-scoring responses (too verbose, missing key info, wrong format)
- Best practices for prompt engineering (clear instructions, constraints, examples)

### How Auto-Tune Works

```
Your Prompt + Dataset Results
        │
        ▼
┌─────────────────────┐
│    Auto-Tune Engine  │
│  Analyzes failures   │
│  Identifies patterns │
│  Rewrites prompt     │
└─────────┬───────────┘
          │
          ▼
   Optimized Prompt
```

> **Note:** Auto-Tune is accessed via REST API. If the endpoint is not available in your environment, the cell below will fall back to a manually optimized prompt for demonstration purposes.

In [ ]:
base_url = str(config.api_url).rstrip('/')
headers = {'Galileo-API-Key': config.api_key.get_secret_value()}
project = get_project(name=PROJECT_NAME)
project_id_str = str(project.id)

# Attempt Auto-Tune via REST API
optimized_system_prompt = None
try:
    auto_tune_response = requests.post(
        f'{base_url}/v2/projects/{project_id_str}/prompts/optimize',
        headers=headers,
        json={
            'prompt': 'Answer the question accurately in 1-2 sentences.',
            'dataset_id': str(dataset_id),
            'preset': 'concise_and_accurate',
        },
        timeout=60,
    )
    auto_tune_response.raise_for_status()
    result = auto_tune_response.json()
    optimized_system_prompt = result.get('optimized_prompt', result.get('prompt'))
    print(f'Auto-Tune succeeded!')
    print(f'Optimized prompt: {optimized_system_prompt}')
except Exception as e:
    print(f'Auto-Tune API not available ({type(e).__name__}): using manual optimization as fallback.')
    optimized_system_prompt = (
        'You are a precise scientific educator. Answer the question in exactly 1-2 sentences. '
        'Focus on the core mechanism or principle. Use specific technical terms. '
        'Do not include examples or analogies unless they add clarity.'
    )
    print(f'Fallback prompt: {optimized_system_prompt}')

## 4. Compare Results — Optimized vs. Baseline

Now we run the same dataset through the **optimized prompt** and compare results against the baseline. This is the core of the Auto-Tune workflow:

1. Run experiment with optimized prompt
2. Compare metric scores side-by-side

In the Galileo Console, you can do this comparison visually in the **Experiments** tab — select both runs and view per-row differences.

In [ ]:
OPTIMIZED_TEMPLATE_NAME = 'auto-tune-optimized-template'

optimized_template = templates.get(name=OPTIMIZED_TEMPLATE_NAME)
if optimized_template is None:
    optimized_template = templates.create(
        name=OPTIMIZED_TEMPLATE_NAME,
        template=[
            Message(role=MessageRole.system, content=optimized_system_prompt),
            Message(role=MessageRole.user, content='{{input}}'),
        ],
        project_name=PROJECT_NAME,
    )

optimized_run = run_experiment(
    experiment_name='auto-tune-optimized',
    dataset=DATASET_NAME,
    prompt_template=optimized_template,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.ground_truth_adherence,
    ],
    prompt_settings=PromptRunSettings(model_alias='GPT-4o mini', temperature=0.3, max_tokens=150),
    project=PROJECT_NAME,
)
optimized_run['message']

## 5. Scorer Feedback — Improve Custom Metrics

**Scorer feedback** lets you tell Galileo when a metric score was wrong. Over time, this feedback improves the accuracy of custom scorers.

### How Scorer Feedback Works

1. You see a metric score that seems wrong (e.g., correctness = 0.3 for a clearly correct answer)
2. You submit feedback: *"This score should be 0.9 because the answer is factually accurate"*
3. Galileo uses this feedback to calibrate the scorer

### Feedback Fields

| Field | Purpose |
|-------|--------|
| `original_score` | The score the metric originally assigned |
| `annotated_score` | What you think the score should be |
| `rationale` | Why you think the original score was wrong |

> **Note:** Scorer feedback is accessed via REST API. If the endpoint is not available, the cell demonstrates the expected pattern.

In [ ]:
# Submit scorer feedback via REST API
feedback_payload = {
    'scorer_name': 'correctness',
    'feedback_items': [
        {
            'original_score': 0.3,
            'annotated_score': 0.9,
            'rationale': 'The answer correctly explains photosynthesis with accurate scientific detail. The low original score appears to be a false negative.',
            'input': 'What is photosynthesis?',
            'output': 'Photosynthesis is the process by which plants convert sunlight, water, and CO2 into glucose and oxygen using chlorophyll.',
        },
        {
            'original_score': 0.4,
            'annotated_score': 0.85,
            'rationale': 'The answer accurately describes the core mechanism of neural network learning. Score should be higher.',
            'input': 'Explain how a neural network learns.',
            'output': 'A neural network learns by adjusting connection weights through backpropagation to minimize prediction errors on training data.',
        },
    ],
}

try:
    feedback_response = requests.post(
        f'{base_url}/v2/projects/{project_id_str}/scorers/feedback',
        headers=headers,
        json=feedback_payload,
        timeout=30,
    )
    feedback_response.raise_for_status()
    print(f'Scorer feedback submitted successfully: {feedback_response.json()}')
except Exception as e:
    print(f'Scorer feedback API not available ({type(e).__name__}): {e}')
    print('In the Galileo Console, you can submit scorer feedback by clicking a metric score')
    print('and selecting "Submit Feedback" to provide your corrected score and rationale.')

## 6. Re-Run with Optimized Prompt — Final Validation

As a final step, we run the optimized prompt one more time with slightly different settings to confirm that the improvements are consistent and not a fluke.

This mirrors a real-world workflow:
1. **Baseline** → Identify weaknesses
2. **Auto-Tune** → Get an optimized prompt
3. **Compare** → Verify improvement
4. **Validate** → Run again with different settings to confirm robustness
5. **Ship** → Deploy the optimized prompt to production

In [ ]:
validation_run = run_experiment(
    experiment_name='auto-tune-validation',
    dataset=DATASET_NAME,
    prompt_template=optimized_template,
    metrics=[
        GalileoMetrics.correctness,
        GalileoMetrics.instruction_adherence,
        GalileoMetrics.ground_truth_adherence,
        GalileoMetrics.bleu,
        GalileoMetrics.rouge,
    ],
    prompt_settings=PromptRunSettings(model_alias='GPT-4o mini', temperature=0.5, max_tokens=200),
    project=PROJECT_NAME,
)
validation_run['message']

## 7. Clean Up

Delete the evaluation dataset and project. Skip this cell if you want to keep the experiment results in the Galileo Console for further comparison.

In [ ]:
try:
    if dataset_id is not None:
        delete_dataset(id=str(dataset_id))
    else:
        delete_dataset(name=DATASET_NAME, project_name=PROJECT_NAME)
except Exception:
    pass

try:
    delete_project(name=PROJECT_NAME)
except Exception:
    pass

'Cleaned up Auto-Tune demo project'